# Dynamic Pricing Engine V2 — Exploratory Data Analysis

**Dataset status: apparently synthetic; provenance unconfirmed.** Results describe this dataset, not real shopper behaviour.

Ground rules for reading this notebook:
- All findings are **descriptive**. Associations are not causal effects.
- Discount comparisons are **descriptive discount lift** only.
- `Price` is used as recorded; the data does not say whether it is before or after discounts.
- Only one year (2024) is available, so patterns are within-year and cannot be shown to recur annually.
- Waste rate = total wastage / (total units sold + total wastage), always a ratio of sums.

This notebook **reads** the checksum-verified V2 raw data and **displays** results. It writes no files;
saved outputs come from `scripts/run_eda.py`.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "dpe").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from dpe.eda import data, plots
from dpe.eda.pipeline import compute_all
from dpe.eda.report import build_findings, render_markdown

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.3f}".format)
print("Project root:", ROOT.name)
print(data.DATASET_LABEL)

In [ ]:
raw = data.load_raw()      # verifies SHA-256 checksums, reads V2 data/raw only
R = compute_all(raw)       # pure computation; no files written
df = R["df"]
df.head()

## 1. Structure, missing values, duplicates and data quality

In [ ]:
R["quality"]["completeness"], R["quality"]["ranges"]

In [ ]:
R["quality"]["profile"]

In [ ]:
R["quality"]["duplicates"]

In [ ]:
R["quality"]["referential"]

In [ ]:
R["quality"]["waste_cap"], R["quality"]["price_variation"]

In [ ]:
_ = plots.completeness_heatmap(R["quality"]["completeness_matrix"])

## 2. Weekly sales trends and within-year patterns

One year only: these are within-year patterns, not demonstrated annual seasonality.

In [ ]:
_ = plots.weekly_units_wastage(R["weekly"])

In [ ]:
_ = plots.category_index(R["category_index"])

In [ ]:
R["months"][["month", "weeks_in_month", "units", "mean_weekly_units", "waste_rate"]]

In [ ]:
_ = plots.acf_weekly(R["weekly"])
R["ljung_box"]

In [ ]:
R["holiday"]["summary"]

## 3. Product and category performance

In [ ]:
R["categories"]

In [ ]:
_ = plots.category_units_waste(R["categories"])

In [ ]:
_ = plots.top_bottom_products(R["products"])

In [ ]:
_ = plots.pareto_curve(R["products"])

In [ ]:
_ = plots.weekly_units_by_category_box(df)

## 4. Store and regional performance

In [ ]:
R["stores"]

In [ ]:
_ = plots.store_units_per_sqft(R["stores"])

In [ ]:
R["regions"]

In [ ]:
_ = plots.region_comparison(R["regions"])

## 5. Wastage quantities and rates

In [ ]:
print(f"Overall waste rate: {R['overall_waste_rate']:.2%}")
_ = plots.waste_rate_hist(df, R["quality"]["waste_cap"])

In [ ]:
_ = plots.waste_heatmap(R["waste_category_month"])

## 6. Price and discount relationships (descriptive)

Price comparisons are within products. Discount results are descriptive discount lift, not causal effects.

In [ ]:
R["price"]["correlations"]

In [ ]:
_ = plots.price_relationship_chart(R["price"])

In [ ]:
R["discount"]

In [ ]:
_ = plots.discount_levels(R["discount"])

In [ ]:
R["ols_table"]  # conditional association with fixed effects; not causal

## 7. Marketing spend and demand

In [ ]:
R["marketing"]["correlations"]

In [ ]:
_ = plots.marketing_relationship_chart(R["marketing"])

## 8. Weather and demand

Recorded actual weather, not forecasts.

In [ ]:
R["temperature"]["correlations"]

In [ ]:
_ = plots.temperature_relationship_chart(R["temperature"])

In [ ]:
R["rainfall"]["correlations"]

## 9. Shelf life, suppliers and operational differences

In [ ]:
R["operations"]["correlations"]

In [ ]:
_ = plots.shelf_life_waste(R["operations"]["products"])

In [ ]:
_ = plots.supplier_waste(R["operations"]["suppliers"])

In [ ]:
R["operations"]["chilled_stores"]

## 10. Outliers and unusual demand patterns

In [ ]:
{k: R["outliers"][k] for k in ("total_flagged", "total_iqr_flagged", "share_flagged")}

In [ ]:
_ = plots.outliers_by_week(R["outliers"]["by_week"])
R["outliers"]["top"]

## 11. Statistical tests (effect sizes first; Benjamini-Hochberg adjusted)

In [ ]:
R["group_tests"]

## 12. Draft findings

Generated text for review; the final report is written to `docs/EDA_FINDINGS.md` after review.

In [ ]:
from IPython.display import Markdown
Markdown(render_markdown(build_findings(R), run_id="notebook preview"))